# LSTM for Sequence Classification

Recurrent neural networks (RNNs) process sequences one step at a time. This notebook trains a character-level LSTM to classify names by language of origin, illustrating how LSTMs handle variable-length sequences.

## Imports and data preparation

We create a small synthetic dataset of surnames labelled by language of origin (Italian, English, German, Spanish). Each name is encoded as a sequence of character indices. A custom `Dataset` and `collate_fn` handle variable-length names by padding them to the same length within each batch.

In [ ]:
import string

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- synthetic dataset of country-style names ---
raw_data = {
    "Italian": ["Romano", "Ricci", "Conti", "Bianchi", "Moretti", "Colombo", "Ferrari", "Esposito", "Rossi", "Bruno",
                "Gallo", "Greco", "Ferrara", "Leone", "Mancini", "Lombardi", "Caruso", "Gentile", "Mariani", "Sorrentino"],
    "English": ["Smith", "Johnson", "Williams", "Brown", "Jones", "Miller", "Davis", "Wilson", "Anderson", "Taylor",
                "Thomas", "Jackson", "White", "Harris", "Martin", "Clark", "Lewis", "Walker", "Young", "King"],
    "German": ["Mueller", "Schmidt", "Schneider", "Fischer", "Weber", "Meyer", "Wagner", "Becker", "Hoffmann", "Koch",
               "Richter", "Klein", "Wolf", "Neumann", "Schwarz", "Braun", "Zimmermann", "Krueger", "Hartmann", "Lange"],
    "Spanish": ["Garcia", "Rodriguez", "Martinez", "Lopez", "Gonzalez", "Hernandez", "Perez", "Sanchez", "Ramirez", "Torres",
                "Flores", "Rivera", "Gomez", "Diaz", "Morales", "Reyes", "Jimenez", "Ruiz", "Alvarez", "Romero"],
}

langs = sorted(raw_data.keys())
lang2idx = {l: i for i, l in enumerate(langs)}
chars = string.ascii_lowercase
char2idx = {c: i + 1 for i, c in enumerate(chars)}  # 0 reserved for padding


def encode_name(name):
    return torch.tensor([char2idx.get(c, 0) for c in name.lower()], dtype=torch.long)


class NamesDataset(Dataset):
    def __init__(self, data, lang2idx):
        self.samples = [(name, lang2idx[lang]) for lang, names in data.items() for name in names]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        name, label = self.samples[idx]
        return encode_name(name), label


def collate(batch):
    seqs, labels = zip(*batch)
    padded = pad_sequence(seqs, batch_first=True, padding_value=0)
    return padded, torch.tensor(labels, dtype=torch.long)


dataset = NamesDataset(raw_data, lang2idx)
loader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=collate)

print(f"Samples: {len(dataset)} | Languages: {langs} | Vocab size: {len(char2idx) + 1}")

## Define LSTM model

The model has three components: an **embedding layer** that converts character indices into dense vectors, an **LSTM layer** that processes the sequence step by step and outputs a final hidden state summarising the entire name, and a **fully-connected layer** that maps the hidden state to class logits.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        _, (hidden, _) = self.lstm(emb)
        return self.fc(hidden.squeeze(0))

model = LSTMClassifier(
    vocab_size=len(char2idx) + 1,
    embed_dim=32,
    hidden_dim=64,
    num_classes=len(langs),
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)
loss_fn = nn.CrossEntropyLoss()

model

## Train and evaluate

We train for 50 epochs on this small dataset. Because the dataset is tiny, we measure training accuracy directly. Every 10 epochs we print the metrics to track progress.

In [ ]:
history = {"epoch": [], "loss": [], "accuracy": []}

for epoch in range(1, 51):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for seqs, labels in loader:
        seqs, labels = seqs.to(device), labels.to(device)
        logits = model(seqs)
        loss = loss_fn(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    acc = correct / total
    history["epoch"].append(epoch)
    history["loss"].append(avg_loss)
    history["accuracy"].append(acc)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}: loss={avg_loss:.4f}, accuracy={acc:.3f}")

## Training curves

We plot loss and accuracy over all 50 epochs. On a small dataset like this, the model should converge quickly to high accuracy.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.plot(history["epoch"], history["loss"])
ax1.set(xlabel="Epoch", ylabel="Loss", title="Training loss")
ax2.plot(history["epoch"], history["accuracy"], color="green")
ax2.set(xlabel="Epoch", ylabel="Accuracy", title="Training accuracy")
plt.tight_layout()
plt.show()

## Predict on new names

We test the trained model on unseen names to see if it can generalise the character patterns it learned to predict the language of origin.

In [ ]:
model.eval()
test_names = ["Rosetti", "Thompson", "Schneider", "Fernandez"]

for name in test_names:
    seq = encode_name(name).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(seq).argmax(1).item()
    print(f"{name:15s} -> {langs[pred]}")

## Practical note

LSTMs process sequences element by element and keep a hidden state that captures long-range dependencies. They are a good first choice for sequence classification, time-series, and language tasks. For very long sequences, attention-based models (Transformers) often perform better.